# Smartphone Feature–Price Analysis

**Purpose:** Clean the smartphone dataset, prepare price bands, and build a baseline regression model for phones priced below ₹20,000.

Detailed feature comparisons are documented in [`sql/analysis_queries.sql`](../sql/analysis_queries.sql).

**Dataset:** Kaggle – Smartphone Models Dataset (`smartphone_cleaned_v5.csv`). Upload the CSV when prompted. The raw dataset is not included in this repository.

## 1. Import libraries and load data

In [ ]:
import numpy as np
import pandas as pd

from google.colab import files

uploaded = files.upload()
file_name = next(iter(uploaded))
df = pd.read_csv(file_name)

print("Dataset shape:", df.shape)
display(df.head())

## 2. Inspect data quality

In [ ]:
quality_check = pd.DataFrame({
    "data_type": df.dtypes.astype(str),
    "missing_values": df.isna().sum(),
    "missing_pct": (df.isna().mean() * 100).round(2)
})

display(quality_check)

print("Exact duplicate rows:", df.duplicated().sum())
print(
    "Repeated brand/model combinations:",
    df.duplicated(subset=["brand_name", "model"]).sum()
)

## 3. Clean and prepare dataset

Missing numeric specifications are retained as missing unless an availability flag confirms that a feature is unavailable. Seven models are flagged for manual verification and excluded from the primary analysis.

In [ ]:
df_clean = df.copy()

# Set feature values to zero only when the dataset confirms they are unavailable.
df_clean.loc[df_clean["fast_charging_available"] == 0, "fast_charging"] = 0
df_clean.loc[df_clean["extended_memory_available"] == 0, "extended_upto"] = 0

# Keep missing category labels visible rather than dropping those records.
df_clean["processor_brand"] = df_clean["processor_brand"].fillna("Unknown")
df_clean["os"] = df_clean["os"].fillna("Unknown")

# Flag records that need manual source verification.
review_models = [
    "Tesla Pi Phone",
    "Apple iPhone 14 Mini",
    "Apple iPhone 15 Ultra",
    "Samsung Galaxy Note 30 Ultra 5G",
    "Nokia N73 5G",
    "Nokia X60 5G",
    "Nokia X60 Pro 5G"
]

df_clean["needs_review"] = df_clean["model"].isin(review_models)

# Create consistent price bands.
price_bins = [0, 10000, 15000, 20000, 30000, 50000, 100000, float("inf")]
price_labels = [
    "Below 10K", "10K-15K", "15K-20K", "20K-30K",
    "30K-50K", "50K-1L", "Above 1L"
]

df_clean["price_band"] = pd.cut(
    df_clean["price"],
    bins=price_bins,
    labels=price_labels,
    right=False
)

review_records = df_clean[df_clean["needs_review"]].copy()
analysis_df = df_clean[~df_clean["needs_review"]].copy()

print("Cleaned dataset:", df_clean.shape)
print("Records flagged for review:", len(review_records))
print("Analysis dataset:", analysis_df.shape)
print("Phones below ₹20,000:", (analysis_df["price"] < 20000).sum())

## 4. Price-band overview

In [ ]:
price_summary = (
    analysis_df.groupby("price_band", observed=False)
    .agg(
        phone_count=("price", "size"),
        avg_price=("price", "mean"),
        median_price=("price", "median"),
        min_price=("price", "min"),
        max_price=("price", "max")
    )
    .reset_index()
)

price_summary[["avg_price", "median_price", "min_price", "max_price"]] = (
    price_summary[["avg_price", "median_price", "min_price", "max_price"]]
    .round(0)
    .astype("Int64")
)
price_summary["dataset_share_pct"] = (
    price_summary["phone_count"] / len(analysis_df) * 100
).round(1)

display(price_summary)

## 5. Multiple linear regression

The model estimates listed price using selected specifications. It uses an 80/20 train-test split and compares its error with a simple average-price baseline. `price_band` and `rating` are excluded to avoid target leakage and reliance on an unverified rating scale.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# Focus on the stakeholder's target segment.
reg_df = analysis_df[analysis_df["price"] < 20000].copy()

numeric_features = [
    "ram_capacity",
    "internal_memory",
    "battery_capacity",
    "processor_speed",
    "primary_camera_rear",
    "refresh_rate"
]
categorical_features = ["processor_brand"]
binary_features = ["has_5g"]
all_features = numeric_features + categorical_features + binary_features

X = reg_df[all_features].copy()
y = reg_df["price"]

# Convert the Boolean 5G flag to numeric form for the model.
X["has_5g"] = X["has_5g"].astype(float)

numeric_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", drop="first"))
])

binary_pipeline = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent"))
])

preprocessor = ColumnTransformer(transformers=[
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
    ("binary", binary_pipeline, binary_features)
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

regression_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("regressor", LinearRegression())
])

regression_model.fit(X_train, y_train)
y_pred = regression_model.predict(X_test)

baseline_pred = np.full(len(y_test), y_train.mean())

results = pd.DataFrame({
    "Metric": ["R2 Score", "MAE", "RMSE", "Baseline MAE"],
    "Value": [
        round(r2_score(y_test, y_pred), 3),
        round(mean_absolute_error(y_test, y_pred), 2),
        round(np.sqrt(mean_squared_error(y_test, y_pred)), 2),
        round(mean_absolute_error(y_test, baseline_pred), 2)
    ]
})

print("Total phones:", len(reg_df))
print("Training records:", len(X_train))
print("Testing records:", len(X_test))
display(results)

## 6. Regression coefficients

In [ ]:
feature_names = regression_model.named_steps["preprocessor"].get_feature_names_out()
coefficients = regression_model.named_steps["regressor"].coef_

coefficient_table = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients
})

coefficient_table["Coefficient"] = coefficient_table["Coefficient"].round(2)
display(coefficient_table.sort_values("Coefficient", ascending=False))

## 7. Key interpretation and limitations

- Use the SQL analysis file for the detailed RAM, storage, 5G, battery, processor, and camera comparisons.
- Regression coefficients are associations, not causal effects. Coefficients use different units and should not be ranked directly by magnitude.
- The rating field's meaning and scale are unverified; it is not used as a regression predictor.
- The model uses one train-test split and a limited feature set. Its metrics are an initial assessment, not guaranteed pricing accuracy.
- Prices are dataset-listed prices. Transaction prices, component costs, demand, and sales volume are outside this analysis.

**Preliminary recommendation:** Evaluate an 8GB RAM / 128GB storage / 5G smartphone with a 5,000 mAh-class battery and a capable Snapdragon or Dimensity chipset, with an indicative target price of ₹18,999. Validate component costs and real-world product quality before final pricing.